# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mr-PeterMaged/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content page (`content_hash_id`), scoped within its `client_hash_id`, drawn from a
single mid-panel month: `fact_content_daily_performance`, partition `month=2026-03`.** Every page is
scored twice from that same partition — a **first-half window** (2026-03-01 to 2026-03-15) that supplies
every feature, and a **second-half window** (2026-03-16 to 2026-03-31) that supplies the label. That
split gives a genuine past-window-predicts-future-window setup without joining across monthly
partitions yet. The full capstone version flagged in w01/w02 (prior 90 days -> next 30 days, across
partitions) is bigger than this task; this is its smallest honest version, built on one mid-panel month
exactly as the data skill recommends — never on `fact_content_daily_performance_sample`, which is the
panel's final month (June 2026) and would make the "future" window the same window a label would later
be tested against.

**Table used:** `fact_content_daily_performance` (grain: `report_date x client_hash_id x
content_hash_id`), March 2026 partition only — verified below. `dim_content` / `dim_clients` are not
joined in this notebook; all five features come from the daily fact table alone.

**Time window:** `report_date` between `2026-03-01` and `2026-03-31` inclusive, split at the 15th/16th.
March (not January) gives every client's March-start history time to accrue; March (not June) keeps the
final month sealed as a future test month, per the data skill's panel warning.

In [1]:
# Section 1 has no query of its own -- the grain and date-span claims above
# are verified together in section 3 (queries Q1 and Q2).

## 2. Fields: feature / label / context / excluded

**Feature (all aggregated over the first-half window, 2026-03-01 to 2026-03-15, `gsc_data_available IS
TRUE` rows only):**
- `impressions_first_half` -- summed `gsc_impressions`
- `clicks_first_half` -- summed `gsc_clicks`
- `avg_position_first_half` -- averaged `gsc_avg_position`
- `ga4_engaged_sessions_first_half` -- summed `ga4_engaged_sessions`, counted only on rows where
  `ga4_data_available IS TRUE` (see Excluded below for why this matters)
- `days_active_first_half` -- count of distinct `report_date` with `gsc_data_available IS TRUE` in the
  first half (a coverage/freshness signal, not a performance one)

**Label / proxy:** `is_declining_label` = 1 if second-half (`2026-03-16`-`2026-03-31`) summed
`gsc_clicks` is lower than first-half summed `gsc_clicks`, else 0 -- restricted to pages with
`clicks_first_half > 0` (a page with zero visibility in the feature window can't honestly be scored
"declining"). This is a **same-shape proxy** to the starter CSV's `trend_direction` trap: it is an
observed within-month comparison, not a model-independent ground truth, and `clicks_second_half` /
`total_month_clicks` must never appear as a feature (that's the deliberate trap in section 3).

**Context (grouping/joining only, never features):** `client_hash_id`, `content_hash_id`.

**Excluded, each with a reason:**
- `gsc_avg_position` on rows where `gsc_data_available IS FALSE` -- placeholder value, not a real rank.
- Any `ga4_*` column on rows where `ga4_data_available IS FALSE` -- zero-filled by the pipeline, not
  "zero engagement" (the data skill's panel warning); only counted when the flag is TRUE.
- `clicks_second_half` / `total_month_clicks` as a feature -- computed from the label window itself;
  including it is exactly the leak this notebook demonstrates and removes in section 3.
- `report_date` as a raw feature -- with the scope limited to one month, a literal date column would
  trivially encode which half of the split a row falls in.

In [2]:
# Section 2 is a classification, not a query -- verified structurally in section 3
# (the feature frame only ever selects the five Feature columns listed above).

## 3. Verify it with queries (grain, counts, availability) + five features + the trap

Three small queries on `month=2026-03`, then the five-feature frame, then the deliberate leak.

In [3]:
import os
import getpass
import duckdb
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Token order: env var -> Colab Secret -> prompt (last resort).
# Never paste the token into a cell -- this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}')")
# Single-threaded on purpose: DuckDB's parallel SUM/AVG combines partial sums in a
# thread-scheduling-dependent order, and floating-point addition isn't associative --
# left multi-threaded, avg_position_first_half (and the honest ROC-AUC computed from it)
# drifted by a thousandth or two between identical runs. One thread trades a little speed
# for exact reproducibility, which matters more here than raw scan speed.
con.execute("SET threads TO 1")

MARCH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

# --- Q1: grain -- one row really is one (report_date, client, content) ---
q1 = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM '{MARCH}'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Q1 -- grain probe (expect 0 rows back):")
print(q1)
assert len(q1) == 0, "grain claim failed: duplicate (date, client, content) rows found"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Q1 -- grain probe (expect 0 rows back):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


In [4]:
# --- Q2: row count + date span for the March partition ---
q2 = con.execute(f"""
    SELECT
      COUNT(*) AS n_rows,
      COUNT(DISTINCT client_hash_id) AS n_clients,
      COUNT(DISTINCT content_hash_id) AS n_content_items,
      MIN(report_date) AS min_date,
      MAX(report_date) AS max_date
    FROM '{MARCH}'
""").df()
print("Q2 -- row count and date span:")
print(q2.to_string(index=False))

Q2 -- row count and date span:
 n_rows  n_clients  n_content_items   min_date   max_date
9841378         55           331437 2026-03-01 2026-03-31


In [5]:
# --- Q3: availability, filtered with IS TRUE ---
q3 = con.execute(f"""
    SELECT
      COUNT(*) AS total_rows,
      SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
      SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM '{MARCH}'
""").df()
total = q3["total_rows"][0]
gsc_avail = q3["gsc_available_rows"][0]
ga4_avail = q3["ga4_available_rows"][0]
print("Q3 -- availability (IS TRUE filter):")
print(q3.to_string(index=False))
print(f"\ngsc_data_available share: {gsc_avail/total:.1%}  ({int(gsc_avail):,} / {int(total):,} rows survive)")
print(f"ga4_data_available share: {ga4_avail/total:.1%}  ({int(ga4_avail):,} / {int(total):,} rows survive)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Q3 -- availability (IS TRUE filter):
 total_rows  gsc_available_rows  ga4_available_rows
    9841378           3611061.0            413966.0

gsc_data_available share: 36.7%  (3,611,061 / 9,841,378 rows survive)
ga4_data_available share: 4.2%  (413,966 / 9,841,378 rows survive)


**Findings so far:** the grain holds (Q1 returns 0 duplicate rows). The March partition has
**9,841,378 rows** across **55 clients** and **331,437 content items**, spanning exactly
`2026-03-01` to `2026-03-31` (Q2) — matching the calendar month claimed in section 1. Availability is
the real story (Q3): only **36.7%** of rows have `gsc_data_available = TRUE`
(3,611,061 / 9,841,378), and only **4.2%** have `ga4_data_available = TRUE` (413,966 / 9,841,378) —
most rows in this table are placeholder rows for a (client, content, day) with no real signal that
day. Every feature below filters on these flags, not on the raw columns.

In [6]:
# --- Five-feature frame: first half (1-15) predicts second half (16-31) ---
frame = con.execute(f"""
    WITH base AS (
        SELECT * FROM '{MARCH}'
        WHERE gsc_data_available IS TRUE
    ),
    first_half AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_impressions) AS impressions_first_half,
            SUM(gsc_clicks) AS clicks_first_half,
            AVG(gsc_avg_position) AS avg_position_first_half,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END)
                AS ga4_engaged_sessions_first_half,
            COUNT(DISTINCT report_date) AS days_active_first_half
        FROM base
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_clicks) AS clicks_second_half
        FROM base
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT f.*, COALESCE(s.clicks_second_half, 0) AS clicks_second_half
    FROM first_half f
    LEFT JOIN second_half s USING (client_hash_id, content_hash_id)
    WHERE f.clicks_first_half > 0
""").df()

frame["is_declining_label"] = (frame["clicks_second_half"] < frame["clicks_first_half"]).astype(int)

# DuckDB doesn't guarantee row order without ORDER BY, so without this sort the frame's row
# order (and therefore train_test_split's shuffle, even with a fixed random_state) can differ
# slightly between runs. Sorting by the grain key makes this notebook's numbers reproducible.
frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

print("feature frame shape:", frame.shape)
print("label positive rate (declining):", f"{frame['is_declining_label'].mean():.1%}")
print(frame["is_declining_label"].value_counts())
frame[["client_hash_id", "content_hash_id", "impressions_first_half", "clicks_first_half",
       "avg_position_first_half", "ga4_engaged_sessions_first_half", "days_active_first_half",
       "is_declining_label"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature frame shape: (51906, 9)
label positive rate (declining): 55.8%
is_declining_label
1    28989
0    22917
Name: count, dtype: int64


,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,ga4_engaged_sessions_first_half,days_active_first_half,is_declining_label
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119.0,1.0,12.639599,0.0,15,0
1,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,440.0,2.0,11.741716,0.0,15,0
2,client_0797ff3a1fc9a6a5,content_9d5a482ec16ea617,54.0,1.0,9.820068,0.0,14,1
3,client_0797ff3a1fc9a6a5,content_a08f993faabe02ea,68.0,2.0,8.350909,0.0,12,1
4,client_0797ff3a1fc9a6a5,content_a7a5f0d74ec03ce8,1621.0,14.0,11.086430,0.0,15,1


**Five features, each with an "available when?" line:**

1. `impressions_first_half` — known by 2026-03-16 (sum of GSC impressions already logged for
   `report_date <= 2026-03-15`).
2. `clicks_first_half` — known by 2026-03-16, same window.
3. `avg_position_first_half` — known by 2026-03-16 (average of `gsc_avg_position` over the same
   `gsc_data_available` days).
4. `ga4_engaged_sessions_first_half` — known by 2026-03-16, counted only where `ga4_data_available
   IS TRUE`, so it is 0 (not missing) for pages/clients without real GA4 access rather than a silent
   zero-fill artifact.
5. `days_active_first_half` — known by 2026-03-16 (a coverage signal: how many of the 15 first-half
   days actually had usable GSC data for this page).

The feature frame has **51,906 pages** with `clicks_first_half > 0`, of which **55.8%** carry the
`is_declining_label` proxy — close to a coin flip, which already hints the proxy will be a weak,
noisy target (confirmed by the honest score below).

In [7]:
# --- The trap: add ONE column derived from the label window, on purpose ---
frame["total_month_clicks"] = frame["clicks_first_half"] + frame["clicks_second_half"]

honest_features = ["impressions_first_half", "clicks_first_half", "avg_position_first_half",
                    "ga4_engaged_sessions_first_half", "days_active_first_half"]
leaked_features = honest_features + ["total_month_clicks"]

results = {}
for label, feats in [("HONEST (first-half only)", honest_features),
                      ("LEAKED (+ total_month_clicks)", leaked_features)]:
    X = frame[feats].fillna(0)
    y = frame["is_declining_label"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    pred = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, pred)
    results[label] = auc
    print(f"{label}: ROC-AUC = {auc:.3f}")

leak_delta = results["LEAKED (+ total_month_clicks)"] - results["HONEST (first-half only)"]
print(f"\nAUC jump from the leak: {leak_delta:.3f}")

HONEST (first-half only): ROC-AUC = 0.547


LEAKED (+ total_month_clicks): ROC-AUC = 1.000

AUC jump from the leak: 0.453


**Trap result:** the honest, first-half-only model scores **ROC-AUC = 0.547** — barely above chance,
an honest signal that this within-month proxy is weak on its own (consistent with w02's finding that no
single signal cleanly separates the label). Adding `total_month_clicks` — a column that literally
contains the second-half clicks the label is computed from — pushes the score to **ROC-AUC = 1.000**.
That is not a better model; it is the label leaking back in through a feature that could only be known
*after* the decision point. `total_month_clicks` is deleted before any real feature set is kept — the
honest number is **0.547**, not 1.000, and that's the number this contract stands behind.

## 4. Data limits

**Named limitation: GA4 coverage is too thin in March 2026 to trust as a feature source on its own.**
Only 4.2% of rows in the March partition have `ga4_data_available = TRUE` (413,966 / 9,841,378, from
Q3) — meaning `ga4_engaged_sessions_first_half` is a real, non-zero signal for a small minority of pages
and a correctly-zeroed-but-uninformative column for almost everyone else. Combined with GSC itself only
being available on 36.7% of rows, this table is mostly placeholder rows for (client, content, day)
combinations with no real activity that day, not a dense daily panel. A capstone version of this
contract should either restrict to the subset of clients with real GA4 access (`dim_clients.
has_ga4_access`) and model that as a client-level segment, or drop GA4 features entirely rather than
average over mostly-missing data. Separately — already flagged in section 2 — the label itself is a
same-month proxy (first half vs. second half), not the true prior-90-day -> next-30-day outcome the
capstone plan calls for; that extension needs a join across monthly partitions, out of scope here.

In [8]:
# Section 4 has no query of its own -- the GA4/GSC availability numbers it cites
# are the same Q3 numbers computed and printed in section 3.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.